In [ ]:
import os
import json
from typing import List, Dict, Any
import numpy as np
from sklearn.cluster import KMeans

# 1. Import genai and its internal client to apply the lifecycle guard
from google import genai
from google.genai import types
from google.genai._api_client import BaseApiClient

# =====================================================================
# MONKEY-PATCH: Prevent google-genai library lifecycle cleanup crashes
# =====================================================================
_original_aclose = BaseApiClient.aclose

async def safe_aclose(self, *args, **kwargs):
    """Safely guards aclose against uninitialized async client properties."""
    if not hasattr(self, '_async_httpx_client'):
        return None
    try:
        return await _original_aclose(self, *args, **kwargs)
    except AttributeError:
        return None

# Bind the safe patch back to the class
BaseApiClient.aclose = safe_aclose
# =====================================================================

# Now initialize your secure vault binding as normal
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception:
    pass

# Client initialization will now execute smoothly without background exceptions
if os.getenv("GCP_PROJECT_ID"):
    client = genai.Client(
        vertexai=True,
        project=os.getenv("GCP_PROJECT_ID"),
        location="us-central1"
    )
else:
    client = genai.Client()


class KanoRouteAgentHarness:
    """
    Secure Agent Harness governing Context-as-a-Perimeter, MCP tool execution,
    and Multimodal Vibe Evaluation for field sales distribution workflows.
    """
    def __init__(self, tenant_id: str):
        self.tenant_id = tenant_id
        self.security_context = f"Tenant-Partition-Key: {tenant_id}"
        
    def execute_vibe_loop(self, user_intent: str, active_skills: List[str]) -> Dict[str, Any]:
        """
        Executes user intent using L1/L2 Progressive Disclosure for Token Economics.
        """
        system_instruction = (
            f"You are KanoRoute AI operating under {self.security_context}. "
            f"Active Skills accessible via progressive disclosure: {json.dumps(active_skills)}. "
            "Enforce Zero Ambient Authority: isolate generated logic execution paths."
        )
        
        # Using the standard stable model for general text generation
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=f"Process field logistics request under strict isolation: {user_intent}",
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.2,
                response_mime_type="application/json"
            )
        )
        return json.loads(response.text)

    def evaluate_visual_alignment(self, screenshot_bytes: bytes, user_spec: str) -> Dict[str, Any]:
        """
        Day 4 Evaluation Axis: Multimodal Judge inspecting the rendered UI artifact.
        """
        prompt = (
            "Score this rendered logistics dashboard interface against the specification "
            "on layout accuracy, regional metric clarity, and button visibility (1-5 scale). "
            "Return output strictly in JSON format."
        )
        
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                prompt,
                user_spec,
                types.Part.from_bytes(data=screenshot_bytes, mime_type="image/png")
            ]
        )
        return json.loads(response.text)

    @staticmethod
    def cluster_field_corrections(raw_traces: List[Dict[str, Any]], openai_client: Any) -> np.ndarray:
        """
        Mines real-time user feedback using OpenAI embeddings and KMeans.
        """
        corrections = []
        for trace in raw_traces:
            for turn in trace.get("turns", []):
                if turn.get("is_correction"):
                    corrections.append(turn.get("user_message"))
                    
        if not corrections:
            return np.array([])
            
        emb_response = openai_client.embeddings.create(
            input=corrections,
            model="text-embedding-3-small"
        )
        vectors = [data.embedding for data in emb_response.data]
        
        num_clusters = min(len(corrections), 4)
        kmeans = KMeans(n_clusters=num_clusters, random_state=42).fit(vectors)
        return kmeans.labels_


# Usage Example for verification pipelines
if __name__ == "__main__":
    harness = KanoRouteAgentHarness(tenant_id="dist-kano-central-001")
    intent = "Optimize weekly biscuit distribution itinerary for Reps A through E inside Kano Municipal grid."
    skills_inventory = ["route_optimization.md", "inventory_reconciliation.md"]
    
    try:
        execution_payload = harness.execute_vibe_loop(intent, skills_inventory)
        print("Secure Agent Intent Trajectory Initialized Successfully:")
        print(json.dumps(execution_payload, indent=2))
    except Exception as e:
        print(f"Harness executed correctly, but model returned error: {e}")
        print("Make sure your Kaggle User Secrets contain the active key under 'GEMINI_API_KEY'.")

In [ ]:
# Simulated follow-up context satisfying the agent's parameter request
refined_intent = """
Optimize weekly biscuit distribution itinerary for Reps A through E inside Kano Municipal grid.
Additional Data provided:
- depot_location: "Kano Central FMCG Warehouse, Zone 4"
- operational_constraints: "Max vehicle capacity 500 crates per Rep. Max 8 hours per shift."
- delivery_points: [
    {"rep": "Rep A", "location": "Kofar Mata Market", "quantity": 150},
    {"rep": "Rep B", "location": "Sabon Gari Market", "quantity": 200},
    {"rep": "Rep C", "location": "Singa Market", "quantity": 120},
    {"rep": "Rep D", "location": "Kwari Market", "quantity": 300},
    {"rep": "Rep E", "location": "Fagge District Center", "quantity": 90}
  ]
"""

# Re-run the loop with the comprehensive context
try:
    final_payload = harness.execute_vibe_loop(refined_intent, skills_inventory)
    print("Final Itinerary Plan Generated Successfully:")
    print(json.dumps(final_payload, indent=2))
except Exception as e:
    print(f"Error during final resolution: {e}")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import json

# Ensure your harness and active inventory from the previous step are initialized
# harness = KanoRouteAgentHarness(tenant_id="dist-kano-central-001")
# skills_inventory = ["route_optimization.md", "inventory_reconciliation.md"]

# 1. UI Elements Setup
title = widgets.HTML("<h2>KanoRoute AI Agent Interface</h2><p>Interact with the safe execution harness.</p>")
intent_input = widgets.Textarea(
    value='Optimize weekly biscuit distribution itinerary for Reps A through E inside Kano Municipal grid.',
    placeholder='Type your logistics or distribution intent here...',
    description='Intent:',
    layout=widgets.Layout(width='90%', height='100px')
)
execute_btn = widgets.Button(
    description='Execute Vibe Loop',
    button_style='primary',
    tooltip='Send intent to the Gemini Harness',
    icon='play'
)
output_area = widgets.Output(layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='10px 0'))

# 2. Event Handler Logic
def on_execute_clicked(b):
    with output_area:
        clear_output()
        user_text = intent_input.value.strip()
        if not user_text:
            print("❌ Error: Please enter a valid intent prompt.")
            return
            
        print("🤖 Communicating with KanoRoute Harness...")
        try:
            # Executes the intent against your secure environment-aware agent
            response_payload = harness.execute_vibe_loop(user_text, skills_inventory)
            
            print("\n✅ Execution Successful! Context-as-a-Perimeter Protected Payload:")
            print(json.dumps(response_payload, indent=2))
            
            # Interactive hook if the agent requests parameters (like our pending_information state)
            if response_payload.get("status") == "pending_information":
                print("\n⚠️ ACTION REQUIRED: Copy the required parameters, update your prompt text above, and run again!")
                
        except Exception as e:
            print(f"❌ Execution Failure: {e}")
            print("Ensure your Kaggle User Secret 'GEMINI_API_KEY' is active and Internet is toggled ON.")

# 3. Bind and Display
execute_btn.on_click(on_execute_clicked)
display(title, intent_input, execute_btn, output_area)